# Barclays Banking Data Quality Checks

This notebook checks **Customers, Accounts, Transactions and Branches** only.

Checks included: data types, missing values, duplicates, primary-key uniqueness, foreign-key validity, date ranges, transaction amounts, account balances, and basic relationship checks.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path('Raw_Data')
OUT_DIR = Path('Barclays Project')
OUT_DIR.mkdir(exist_ok=True)

files = {
    'customers': DATA_DIR / 'customers.csv',
    'accounts': DATA_DIR / 'accounts.csv',
    'transactions': DATA_DIR / 'transactions.csv',
    'branches': DATA_DIR / 'branches.csv'
}


In [3]:
customers = pd.read_csv(files['customers'])
accounts = pd.read_csv(files['accounts'])
transactions = pd.read_csv(files['transactions'])
branches = pd.read_csv(files['branches'])

data = {
    'Customers': customers,
    'Accounts': accounts,
    'Transactions': transactions,
    'Branches': branches
}

for name, df in data.items():
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]} columns')


Customers: 1,111 rows x 6 columns
Accounts: 1,667 rows x 6 columns
Transactions: 50,000 rows x 8 columns
Branches: 50 rows x 3 columns


## 1. Basic profiling

In [4]:
for name, df in data.items():
    print('\n' + '='*70)
    print(name)
    display(df.head())
    print('\nData types:')
    display(df.dtypes.to_frame('dtype'))



Customers


,CustomerID,FirstName,LastName,DateOfBirth,AddressID,CustomerTypeID
0,10832,Nyla,Aguirre,1974-02-07 00:00:00.000000,881,1
1,10983,NaN,Battle,1963-02-01 00:00:00.000000,958,2
2,10837,Angelena,Harrington,1964-03-25 00:00:00.000000,86,3
3,10107,Remona,Glass,1965-09-16 00:00:00.000000,595,1
4,10553,King,Becker,1966-02-20 00:00:00.000000,969,3



Data types:


,dtype
CustomerID,int64
FirstName,str
LastName,str
DateOfBirth,str
AddressID,int64
CustomerTypeID,int64



Accounts


...

,dtype
TransactionID,int64
AccountOriginID,int64
AccountDestinationID,int64
TransactionTypeID,int64
Amount,float64
TransactionDate,str
BranchID,int64
Description,str



Branches


,BranchID,BranchName,AddressID
0,1,Branch 1,733
1,2,Branch 2,511
2,3,Branch 3,27
3,4,Branch 4,97
4,5,Branch 5,796



Data types:


,dtype
BranchID,int64
BranchName,str
AddressID,int64


## 2. Missing values

In [5]:
missing_report = []
for name, df in data.items():
    for col in df.columns:
        missing = int(df[col].isna().sum())
        missing_report.append([name, col, missing, round(missing/len(df)*100, 2)])

missing_report = pd.DataFrame(missing_report, columns=['Dataset','Column','Missing_Count','Missing_Percent'])
display(missing_report[missing_report['Missing_Count'] > 0].sort_values(['Dataset','Missing_Count'], ascending=[True,False]))
missing_report.to_csv(OUT_DIR/'missing_values_report.csv', index=False)


,Dataset,Column,Missing_Count,Missing_Percent
11,Accounts,OpeningDate,33,1.98
2,Customers,LastName,23,2.07
1,Customers,FirstName,22,1.98
17,Transactions,TransactionDate,1000,2.00


## 3. Duplicate records

In [6]:
for name, df in data.items():
    print(f'{name}: {df.duplicated().sum():,} exact duplicate rows')


Customers: 11 exact duplicate rows
Accounts: 16 exact duplicate rows
Transactions: 500 exact duplicate rows
Branches: 0 exact duplicate rows


## 4. Primary-key uniqueness

In [7]:
pk_map = {
    'Customers': 'CustomerID',
    'Accounts': 'AccountID',
    'Transactions': 'TransactionID',
    'Branches': 'BranchID'
}

pk_rows = []
for name, pk in pk_map.items():
    df = data[name]
    if pk not in df.columns:
        pk_rows.append([name, pk, 'COLUMN NOT FOUND', np.nan, np.nan])
        continue
    duplicate_ids = int(df[pk].duplicated(keep=False).sum())
    unique_ids = int(df[pk].nunique(dropna=True))
    pk_rows.append([name, pk, 'PASS' if duplicate_ids == 0 else 'FAIL', unique_ids, duplicate_ids])

pk_report = pd.DataFrame(pk_rows, columns=['Dataset','Primary_Key','Status','Unique_IDs','Rows_with_Duplicate_ID'])
display(pk_report)
pk_report.to_csv(OUT_DIR/'primary_key_report.csv', index=False)


,Dataset,Primary_Key,Status,Unique_IDs,Rows_with_Duplicate_ID
0,Customers,CustomerID,FAIL,1100,22
1,Accounts,AccountID,FAIL,1651,32
2,Transactions,TransactionID,FAIL,49500,1000
3,Branches,BranchID,PASS,50,0


## 5. Foreign-key / relationship checks

In [8]:
def invalid_fk(child_df, child_col, parent_df, parent_col):
    child_values = child_df[child_col].dropna()
    parent_values = set(parent_df[parent_col].dropna())
    return int((~child_values.isin(parent_values)).sum())

fk_results = []

checks = [
    ('Accounts','CustomerID','Customers','CustomerID'),
    ('Accounts','BranchID','Branches','BranchID'),
    ('Transactions','AccountOriginID','Accounts','AccountID'),
    ('Transactions','AccountDestinationID','Accounts','AccountID'),
    ('Transactions','BranchID','Branches','BranchID'),
]

for child, child_col, parent, parent_col in checks:
    if child_col in data[child].columns and parent_col in data[parent].columns:
        bad = invalid_fk(data[child], child_col, data[parent], parent_col)
        fk_results.append([child, child_col, parent, parent_col, bad, 'PASS' if bad == 0 else 'FAIL'])

fk_report = pd.DataFrame(fk_results, columns=['Child_Dataset','Child_Field','Parent_Dataset','Parent_Field','Invalid_Count','Status'])
display(fk_report)
fk_report.to_csv(OUT_DIR/'foreign_key_report.csv', index=False)


,Child_Dataset,Child_Field,Parent_Dataset,Parent_Field,Invalid_Count,Status
0,Accounts,CustomerID,Customers,CustomerID,0,PASS
1,Transactions,AccountOriginID,Accounts,AccountID,0,PASS
2,Transactions,AccountDestinationID,Accounts,AccountID,0,PASS
3,Transactions,BranchID,Branches,BranchID,0,PASS


## 6. Date checks

In [9]:
date_columns = {
    'Customers': ['DateOfBirth'],
    'Accounts': ['OpeningDate'],
    'Transactions': ['TransactionDate']
}

date_report = []
for name, cols in date_columns.items():
    for col in cols:
        if col not in data[name].columns:
            continue
        parsed = pd.to_datetime(data[name][col], errors='coerce')
        date_report.append([
            name, col, int(parsed.isna().sum()),
            parsed.min(), parsed.max()
        ])

date_report = pd.DataFrame(date_report, columns=['Dataset','Column','Invalid_or_Missing','Min_Date','Max_Date'])
display(date_report)
date_report.to_csv(OUT_DIR/'date_report.csv', index=False)

# Future DOB check
if 'DateOfBirth' in customers.columns:
    dob = pd.to_datetime(customers['DateOfBirth'], errors='coerce')
    future_dob = int((dob > pd.Timestamp.today()).sum())
    print('Future DateOfBirth values:', future_dob)


,Dataset,Column,Invalid_or_Missing,Min_Date,Max_Date
0,Customers,DateOfBirth,32,1960-01-27,2026-07-06 15:01:42.854835
1,Accounts,OpeningDate,33,2018-01-03,2026-07-06 15:01:42.900415
2,Transactions,TransactionDate,1000,2020-01-01,2026-08-28 15:01:43.084879


Future DateOfBirth values: 0


## 7. Transaction amount checks

In [10]:
amount_col = next((c for c in ['Amount','TransactionAmount'] if c in transactions.columns), None)
if amount_col:
    amount = pd.to_numeric(transactions[amount_col], errors='coerce')
    print('Missing/non-numeric amounts:', int(amount.isna().sum()))
    print('Zero amounts:', int((amount == 0).sum()))
    print('Negative amounts:', int((amount < 0).sum()))
    print('Minimum amount:', amount.min())
    print('Maximum amount:', amount.max())


Missing/non-numeric amounts: 0
Zero amounts: 0
Negative amounts: 0
Minimum amount: 1.01
Maximum amount: 4999.59


## 8. Account balance checks

In [11]:
balance_col = next((c for c in ['Balance','AccountBalance'] if c in accounts.columns), None)
if balance_col:
    balance = pd.to_numeric(accounts[balance_col], errors='coerce')
    print('Missing/non-numeric balances:', int(balance.isna().sum()))
    print('Zero balances:', int((balance == 0).sum()))
    print('Negative balances:', int((balance < 0).sum()))
    print('Minimum balance:', balance.min())
    print('Maximum balance:', balance.max())


Missing/non-numeric balances: 0
Zero balances: 0
Negative balances: 10
Minimum balance: -486.68
Maximum balance: 99828.98


## 9. Overall data-quality summary

In [12]:
summary = []
for name, df in data.items():
    summary.append([
        name,
        len(df),
        df.shape[1],
        int(df.isna().sum().sum()),
        int(df.duplicated().sum())
    ])

summary = pd.DataFrame(summary, columns=['Dataset','Rows','Columns','Missing_Cells','Exact_Duplicate_Rows'])
display(summary)
summary.to_csv(OUT_DIR/'data_quality_summary.csv', index=False)


,Dataset,Rows,Columns,Missing_Cells,Exact_Duplicate_Rows
0,Customers,1111,6,45,11
1,Accounts,1667,6,33,16
2,Transactions,50000,8,1000,500
3,Branches,50,3,0,0


## Next step

Use the reports above to decide which issues should be corrected, excluded, or flagged for review. We should **not blindly delete negative balances or other unusual values** because they may be legitimate business data.